### create new cache on hard drive

In [1]:
from pathlib import Path
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

drive = Path("/Volumes/WD_BLACK")

cache_dir = drive / "Spikes2Slope" / "allen_visual_coding_data"
cache_dir.mkdir(parents=True, exist_ok=True)

manifest_path = cache_dir / "manifest.json"

coding_cache = EcephysProjectCache.from_warehouse(
    manifest=str(manifest_path)
)

print("Cache directory:", cache_dir)
print("Manifest exists:", manifest_path.exists())
print("Available storage:",
      round(__import__("shutil").disk_usage(drive).free / 1024**3, 2),
      "GiB")


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cache directory: /Volumes/WD_BLACK/Spikes2Slope/allen_visual_coding_data
Manifest exists: True
Available storage: 307.0 GiB


### identify optotagging-eligible sessions

In [2]:
import numpy as np
import pandas as pd

coding_sessions = coding_cache.get_session_table()

opto_sessions = (
    coding_sessions
    .rename_axis("ecephys_session_id")
    .reset_index()
    [["ecephys_session_id", "specimen_id", "full_genotype"]]
    .copy()
)

genotype = opto_sessions["full_genotype"].fillna("")

opto_sessions["cre_line"] = np.select(
    [
        genotype.str.startswith("Pvalb"),
        genotype.str.startswith("Sst"),
        genotype.str.startswith("Vip")
    ],
    ["Pvalb", "Sst", "Vip"],
    default="other"
)

eligible = opto_sessions.loc[
    (opto_sessions["cre_line"] != "other")
    & genotype.str.contains("Ai32", regex=False)
].copy()

eligible = eligible.sort_values(
    ["cre_line", "ecephys_session_id"]
).reset_index(drop=True)

assert len(eligible) == 28, "Expected 28 eligible sessions"
assert eligible["ecephys_session_id"].is_unique

print("Eligible sessions:", len(eligible))
print("\nSessions by Cre line:")
print(eligible["cre_line"].value_counts().sort_index())

display(eligible)

eligible.to_csv(
    cache_dir.parent / "eligible_opto_sessions.csv",
    index=False
)


Eligible sessions: 28

Sessions by Cre line:
Pvalb     8
Sst      12
Vip       8
Name: cre_line, dtype: int64


,ecephys_session_id,specimen_id,full_genotype,cre_line
0,721123822,707296982,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
1,746083955,726170935,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
2,760345702,739783171,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
3,773418906,757329624,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
4,797828357,776061251,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
5,829720705,811322619,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
6,839557629,821469666,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
7,840012044,820866121,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
8,715093703,699733581,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Sst
9,719161530,703279284,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Sst


### the big download

In [3]:

import gc
import time

download_log_path = cache_dir.parent / "opto_download_log.csv"

if download_log_path.exists():
    download_log = pd.read_csv(download_log_path)
else:
    download_log = pd.DataFrame(columns=[
        "ecephys_session_id", "cre_line",
        "status", "error", "elapsed_seconds"
    ])

for row in eligible.itertuples(index=False):
    session_id = int(row.ecephys_session_id)

    completed = download_log.loc[
        (download_log["ecephys_session_id"] == session_id)
        & (download_log["status"] == "success")
    ]

    if not completed.empty:
        print(f"Skipping {session_id}: already downloaded")
        continue

    print(f"\nDownloading session {session_id} ({row.cre_line})...")
    start = time.time()

    try:
        session = coding_cache.get_session_data(session_id)

        epochs = session.optogenetic_stimulation_epochs
        n_units = len(session.units)

        status = "success"
        error = ""

        print(
            f"Success: {session_id} | "
            f"{n_units} units | {len(epochs)} opto epochs"
        )

        del session
        gc.collect()

    except Exception as exc:
        status = "failed"
        error = str(exc)
        print(f"Failed: {session_id} | {error}")

    elapsed = round(time.time() - start, 1)

    download_log = download_log[
        download_log["ecephys_session_id"] != session_id
    ]

    download_log = pd.concat([
        download_log,
        pd.DataFrame([{
            "ecephys_session_id": session_id,
            "cre_line": row.cre_line,
            "status": status,
            "error": error,
            "elapsed_seconds": elapsed
        }])
    ], ignore_index=True)

    download_log.to_csv(download_log_path, index=False)

print("\nDownload summary:")
display(download_log["status"].value_counts())

print("\nLog saved to:", download_log_path)


Skipping 721123822: already downloaded
Skipping 746083955: already downloaded
Skipping 760345702: already downloaded
Skipping 773418906: already downloaded
Skipping 797828357: already downloaded
Skipping 829720705: already downloaded
Skipping 839557629: already downloaded
Skipping 840012044: already downloaded
Skipping 715093703: already downloaded
Skipping 719161530: already downloaded
Skipping 756029989: already downloaded
Skipping 758798717: already downloaded
Skipping 760693773: already downloaded
Skipping 762602078: already downloaded
Skipping 786091066: already downloaded
Skipping 787025148: already downloaded
Skipping 789848216: already downloaded
Skipping 794812542: already downloaded
Skipping 831882777: already downloaded
Skipping 839068429: already downloaded
Skipping 751348571: already downloaded
Skipping 755434585: already downloaded
Skipping 762120172: already downloaded
Skipping 791319847: already downloaded
Skipping 798911424: already downloaded
Skipping 816200189: alrea

success    28
Name: status, dtype: int64


Log saved to: /Volumes/WD_BLACK/Spikes2Slope/opto_download_log.csv


### successfully downloaded now to inspect optogenetic stimulation protocols

In [6]:

import gc

protocol_results = []

for row in eligible.itertuples(index=False):
    session_id = int(row.ecephys_session_id)
    session = coding_cache.get_session_data(session_id)
    epochs = session.optogenetic_stimulation_epochs

    pulses = epochs.loc[
        (epochs["stimulus_name"] == "pulse")
        & epochs["duration"].between(
            0.009, 0.020, inclusive="neither"
        )
    ]

    protocol_results.append({
        "ecephys_session_id": session_id,
        "cre_line": row.cre_line,
        "total_epochs": len(epochs),
        "ten_ms_pulses": len(pulses),
        "light_levels": sorted(pulses["level"].dropna().unique())
    })

    del session
    gc.collect()

protocol_summary = pd.DataFrame(protocol_results)

display(protocol_summary)

print("Sessions analyzed:", len(protocol_summary))
print("Sessions with 10 ms pulses:",
      (protocol_summary["ten_ms_pulses"] > 0).sum())

print("Sessions without 10 ms pulses:",
      (protocol_summary["ten_ms_pulses"] == 0).sum())

protocol_summary.to_csv(
    cache_dir.parent / "opto_protocol_summary.csv",
    index=False
)


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

,ecephys_session_id,cre_line,total_epochs,ten_ms_pulses,light_levels
0,721123822,Pvalb,180,45,"[1.0, 2.5, 4.0]"
1,746083955,Pvalb,180,45,"[1.0, 2.5, 4.0]"
2,760345702,Pvalb,180,45,"[1.0, 2.5, 4.0]"
3,773418906,Pvalb,180,45,"[1.0, 2.5, 4.0]"
4,797828357,Pvalb,300,75,"[1.3, 1.7, 2.0]"
5,829720705,Pvalb,300,75,"[1.3, 1.7, 2.0]"
6,839557629,Pvalb,300,75,"[1.3, 1.7, 2.0]"
7,840012044,Pvalb,300,75,"[0.6375, 0.7375, 0.82]"
8,715093703,Sst,180,45,"[1.0, 2.5, 4.0]"
9,719161530,Sst,180,45,"[1.0, 2.5, 4.0]"


Sessions analyzed: 28
Sessions with 10 ms pulses: 28
Sessions without 10 ms pulses: 0


### optotagging responses across sessions

In [5]:

visual_regions = [
    "VISp", "VISam", "VISrl", "VISal",
    "VISl", "VISpm", "LGd"
]

results_dir = cache_dir.parent / "opto_results"
results_dir.mkdir(exist_ok=True)

all_results = []
processing_log = []

for row in eligible.itertuples(index=False):
    session_id = int(row.ecephys_session_id)
    result_path = results_dir / f"opto_units_{session_id}.csv"

    if result_path.exists():
        all_results.append(pd.read_csv(result_path))
        print(f"Loaded saved results: {session_id}")
        continue

    try:
        session = coding_cache.get_session_data(session_id)
        epochs = session.optogenetic_stimulation_epochs

        pulses = epochs.loc[
            (epochs["stimulus_name"] == "pulse")
            & epochs["duration"].between(
                0.009, 0.020, inclusive="neither"
            )
        ]

        if pulses.empty:
            raise ValueError("No compatible 10 ms pulses")

        trial_starts = pulses["start_time"].to_numpy()

        units = session.units.loc[
            session.units["ecephys_structure_acronym"].isin(
                visual_regions
            )
        ]

        session_results = []

        for unit_id, unit in units.iterrows():
            spikes = np.asarray(session.spike_times[unit_id])

            baseline_counts = (
                np.searchsorted(spikes, trial_starts - 0.002)
                - np.searchsorted(spikes, trial_starts - 0.010)
            )

            evoked_counts = (
                np.searchsorted(spikes, trial_starts + 0.009)
                - np.searchsorted(spikes, trial_starts + 0.001)
            )

            baseline_rate = baseline_counts.mean() / 0.008
            evoked_rate = evoked_counts.mean() / 0.008

            session_results.append({
                "unit_id": unit_id,
                "ecephys_session_id": session_id,
                "specimen_id": row.specimen_id,
                "cre_line": row.cre_line,
                "region": unit["ecephys_structure_acronym"],
                "baseline_rate_hz": baseline_rate,
                "evoked_rate_hz": evoked_rate,
                "response_ratio": evoked_rate / (baseline_rate + 1),
                "snr": unit["snr"],
                "amplitude_cutoff": unit["amplitude_cutoff"],
                "presence_ratio": unit["presence_ratio"],
                "isi_violations": unit["isi_violations"],
                "n_pulses": len(pulses)
            })

        session_df = pd.DataFrame(session_results)

        session_df["candidate_tagged"] = (
            session_df["response_ratio"] > 2
        )

        session_df["allen_qc"] = (
            (session_df["presence_ratio"] >= 0.95)
            & (session_df["isi_violations"] <= 0.5)
            & (session_df["amplitude_cutoff"] <= 0.1)
        )

        session_df["snr_qc"] = (
            session_df["allen_qc"]
            & (session_df["snr"] >= 1)
        )

        session_df["matched_candidate"] = (
            session_df["candidate_tagged"]
            & session_df["snr_qc"]
        )

        assert session_df["unit_id"].is_unique

        session_df.to_csv(result_path, index=False)
        all_results.append(session_df)

        processing_log.append({
            "ecephys_session_id": session_id,
            "status": "success",
            "units": len(session_df),
            "candidate_tagged": int(
                session_df["candidate_tagged"].sum()
            ),
            "qc_matched": int(
                session_df["matched_candidate"].sum()
            )
        })

        print(
            f"{session_id} ({row.cre_line}): "
            f"{len(session_df)} units, "
            f"{session_df['matched_candidate'].sum()} QC candidates"
        )

    except Exception as exc:
        processing_log.append({
            "ecephys_session_id": session_id,
            "status": "failed",
            "error": str(exc)
        })
        print(f"FAILED {session_id}: {exc}")

    finally:
        if "session" in locals():
            del session
        gc.collect()

opto_results = pd.concat(all_results, ignore_index=True)
processing_log = pd.DataFrame(processing_log)

assert opto_results["unit_id"].is_unique

opto_results.to_csv(
    results_dir / "all_opto_unit_results.csv",
    index=False
)

processing_log.to_csv(
    results_dir / "processing_log.csv",
    index=False
)

print("\nSessions with results:",
      opto_results["ecephys_session_id"].nunique())

print("Total candidate optotagged units:",
      int(opto_results["candidate_tagged"].sum()))

print("QC-matched candidates:",
      int(opto_results["matched_candidate"].sum()))


Loaded saved results: 721123822
Loaded saved results: 746083955
Loaded saved results: 760345702
Loaded saved results: 773418906
Loaded saved results: 797828357
Loaded saved results: 829720705
Loaded saved results: 839557629
Loaded saved results: 840012044
Loaded saved results: 715093703
Loaded saved results: 719161530
Loaded saved results: 756029989
Loaded saved results: 758798717
Loaded saved results: 760693773
Loaded saved results: 762602078
Loaded saved results: 786091066
Loaded saved results: 787025148
Loaded saved results: 789848216
Loaded saved results: 794812542
Loaded saved results: 831882777
Loaded saved results: 839068429


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

751348571 (Vip): 334 units, 44 QC candidates


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

755434585 (Vip): 405 units, 58 QC candidates


...

816200189 (Vip): 253 units, 5 QC candidates


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

819701982 (Vip): 273 units, 20 QC candidates


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

835479236 (Vip): 339 units, 27 QC candidates

Sessions with results: 28
Total candidate optotagged units: 1152
QC-matched candidates: 1127


### optotagged coverage table

In [7]:

visual_regions = [
    "VISp", "VISam", "VISrl", "VISal",
    "VISl", "VISpm", "LGd"
]

assert opto_results["ecephys_session_id"].nunique() == 28
assert opto_results["unit_id"].is_unique

session_region = eligible[
    ["ecephys_session_id", "specimen_id", "cre_line"]
].merge(
    pd.DataFrame({"region": visual_regions}),
    how="cross"
)

unit_counts = (
    opto_results
    .groupby(["ecephys_session_id", "region"])
    .size()
    .rename("total_units")
    .reset_index()
)

tagged_counts = (
    opto_results[opto_results["matched_candidate"]]
    .groupby(["ecephys_session_id", "region"])
    .size()
    .rename("tagged_units")
    .reset_index()
)

session_region = (
    session_region
    .merge(
        unit_counts,
        on=["ecephys_session_id", "region"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        tagged_counts,
        on=["ecephys_session_id", "region"],
        how="left",
        validate="one_to_one"
    )
)

session_region[["total_units", "tagged_units"]] = (
    session_region[["total_units", "tagged_units"]]
    .fillna(0)
    .astype(int)
)

assert len(session_region) == 28 * 7
assert not session_region.duplicated(
    ["ecephys_session_id", "region"]
).any()
assert session_region["tagged_units"].sum() == 1127
assert (
    session_region["tagged_units"]
    <= session_region["total_units"]
).all()

region_totals = (
    session_region
    .groupby("region", as_index=False)
    [["total_units", "tagged_units"]]
    .sum()
)

print("Session-region combinations:", len(session_region))
print("Total QC-matched candidates:",
      session_region["tagged_units"].sum())

print("\nCandidate counts by region:")
display(region_totals)

coverage_matrix = (
    session_region.pivot(
        index="ecephys_session_id",
        columns="region",
        values="tagged_units"
    )
    .reindex(columns=visual_regions)
)

coverage_matrix["n_regions"] = (
    coverage_matrix[visual_regions] > 0
).sum(axis=1)

coverage_matrix["total_tagged"] = (
    coverage_matrix[visual_regions].sum(axis=1)
)

coverage_matrix = coverage_matrix.sort_values(
    ["n_regions", "total_tagged"],
    ascending=False
)

print("\nOptotagged session coverage:")
display(coverage_matrix)

session_region.to_csv(
    results_dir / "session_region_coverage.csv",
    index=False
)

region_totals.to_csv(
    results_dir / "region_totals.csv",
    index=False
)

coverage_matrix.to_csv(
    results_dir / "opto_coverage_matrix.csv"
)


Session-region combinations: 196
Total QC-matched candidates: 1127

Candidate counts by region:


,region,total_units,tagged_units
0,LGd,488,59
1,VISal,1381,217
2,VISam,1545,226
3,VISl,1178,110
4,VISp,1730,226
5,VISpm,991,133
6,VISrl,1266,156



Optotagged session coverage:


region,VISp,VISam,VISrl,VISal,VISl,VISpm,LGd,n_regions,total_tagged
ecephys_session_id,,,,,,,,,
756029989,13,10,5,10,6,19,3,7,66
755434585,16,15,11,3,2,7,4,7,58
719161530,11,5,1,1,7,3,13,7,41
791319847,4,4,4,3,2,3,1,7,21
715093703,7,6,12,0,4,11,15,6,55
794812542,14,18,7,5,7,1,0,6,52
760345702,6,7,0,16,7,5,8,6,49
831882777,6,11,8,8,7,6,0,6,46
797828357,12,8,4,1,3,7,0,6,35


In [8]:

standout_ids = [
    756029989,
    755434585,
    719161530,
    791319847,
    786091066
]

standout_sessions = (
    coverage_matrix
    .loc[standout_ids]
    .reset_index()
    .merge(
        eligible[
            ["ecephys_session_id", "specimen_id", "cre_line"]
        ],
        on="ecephys_session_id",
        how="left",
        validate="one_to_one"
    )
)

standout_sessions = standout_sessions[
    ["ecephys_session_id", "specimen_id", "cre_line"]
    + visual_regions
    + ["n_regions", "total_tagged"]
]

display(standout_sessions)

standout_sessions.to_csv(
    results_dir / "standout_opto_sessions.csv",
    index=False
)


,ecephys_session_id,specimen_id,cre_line,VISp,VISam,VISrl,VISal,VISl,VISpm,LGd,n_regions,total_tagged
0,756029989,734865738,Sst,13,10,5,10,6,19,3,7,66
1,755434585,730760270,Vip,16,15,11,3,2,7,4,7,58
2,719161530,703279284,Sst,11,5,1,1,7,3,13,7,41
3,791319847,769360779,Vip,4,4,4,3,2,3,1,7,21
4,786091066,763884103,Sst,9,12,3,33,0,12,0,5,69
